# Chapter 5 companion: attention, by hand and for real

**LLM Soup to Nuts** &middot; [read the chapter](https://talktovishal.github.io/projects/llm-soup-to-nuts/chapter-5.html)

This notebook does two things.

1. **Rebuilds the chapter's attention by hand in NumPy**, using the exact same sentence and the exact same query, key and value vectors. If the chapter is right, this notebook must reproduce `46.3%` for the `it` &rarr; `cat` pair. We check that automatically.
2. **Loads the real GPT-2 and plots its actual attention maps.** This is the part the browser could not do: the in-browser export only hands back logits and the key/value cache, while PyTorch will happily give us `output_attentions=True`.

Run the cells in order. Everything is free and nothing needs a GPU.

> **Tip.** In Colab, `Runtime &rarr; Run all` does the whole thing in about a minute.

---
## Part 1. Attention by hand

Six tokens, four channels. In a real model these vectors come from multiplying each token's
embedding by the learned matrices `W_Q`, `W_K` and `W_V`. Here they are written out directly so
every number stays readable - the arithmetic that follows is identical either way.

In [ ]:
import numpy as np

TOKENS = ["Sarah", "fed", "the", "cat", "because", "it"]
CHANNELS = ["subject", "verb", "function", "animate"]

Q = np.array([
    [0.0, 1.4, 0.0, 0.2],   # Sarah
    [0.3, 0.0, 0.2, 1.4],   # fed
    [0.0, 0.0, 0.0, 1.4],   # the
    [0.6, 0.6, 0.0, 0.2],   # cat
    [0.0, 0.8, 0.4, 0.0],   # because
    [0.1, 0.0, 0.3, 1.8],   # it
])

K = np.array([
    [1.6, 0.0, 0.0, 0.15],
    [0.0, 1.6, 0.0, 0.0],
    [0.0, 0.0, 1.4, 0.0],
    [0.0, 0.0, 0.0, 1.8],
    [0.0, 0.0, 1.2, 0.0],
    [0.0, 0.0, 0.5, 0.1],
])

V = np.array([
    [1.0, 0.0, 0.0, 0.3],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
    [0.0, 0.0, 0.8, 0.0],
    [0.0, 0.0, 0.3, 0.3],
])

d = Q.shape[1]
print(f"{len(TOKENS)} tokens, d = {d}, so we scale by sqrt(d) = {np.sqrt(d):.0f}")

### Where the query actually comes from

The chapter shows one projection worked out longhand. Here it is in code: the embedding for `it`
goes through `W_Q` and out comes the query vector used everywhere else.

In [ ]:
x_it = np.array([0.2, 0.1, 0.5, 0.9])

W_Q = np.array([
    [0.0, 0.0, 0.5, 0.5],
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.4, 0.7],
    [0.0, 0.0, 0.0, 1.5],
])

q_it = x_it @ W_Q
print("x_it       =", x_it)
print("q_it       =", q_it)
print("chapter q  =", Q[5])
assert np.allclose(q_it, Q[5]), "projection does not match the chapter"
print("\nOne vector in, three vectors out - this is the query half of it.")

### Scaled dot-product attention, in five lines

This is the whole mechanism. Everything else in a Transformer is packaging.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # subtract the max for numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def attention(Q, K, V, causal=False):
    scores = Q @ K.T / np.sqrt(Q.shape[1])
    if causal:
        mask = np.triu(np.ones_like(scores), k=1).astype(bool)
        scores = np.where(mask, -np.inf, scores)
    weights = softmax(scores)
    return weights @ V, weights, scores


out, weights, scores = attention(Q, K, V)

it = TOKENS.index("it")
cat = TOKENS.index("cat")
print(f"'it' attends to 'cat' with weight {weights[it, cat] * 100:.1f}%")
assert abs(weights[it, cat] * 100 - 46.3) < 0.1, "this must match the chapter"
print("Matches the chapter. Good - the page and this notebook agree.")
print(f"\nNew vector for 'it': {np.round(out[it], 2)}")

In [ ]:
import pandas as pd

print("Full attention matrix (rows = query, columns = key), in percent:\n")
display(
    pd.DataFrame(np.round(weights * 100, 1), index=TOKENS, columns=TOKENS)
    .style.background_gradient(cmap="BuGn", axis=None)
    .format("{:.1f}")
)

### The causal mask

Turn it on and the upper triangle disappears. Each row still sums to 100%, because softmax shares
the probability out over whatever is left.

In [ ]:
_, causal_weights, _ = attention(Q, K, V, causal=True)

display(
    pd.DataFrame(np.round(causal_weights * 100, 1), index=TOKENS, columns=TOKENS)
    .style.background_gradient(cmap="BuGn", axis=None)
    .format("{:.1f}")
)
print("Row sums:", np.round(causal_weights.sum(axis=1), 6))

### Why we divide by sqrt(d)

A dot product adds up `d` products, so its spread grows like `sqrt(d)`. Without the division,
softmax saturates as the head gets wider: one token takes everything and the gradients through the
rest vanish. Watch the top weight climb in the unscaled column and sit still in the scaled one.

In [ ]:
base = Q[it] @ K.T / np.sqrt(d)   # the per-sqrt(d) scores from the chapter

rows = []
for width in [4, 16, 64, 256, 1024]:
    unscaled = softmax(base * np.sqrt(width))
    scaled = softmax(base)
    rows.append({
        "d": width,
        "top weight, unscaled": f"{unscaled.max() * 100:.3f}%",
        "top weight, scaled": f"{scaled.max() * 100:.1f}%",
    })

display(pd.DataFrame(rows).set_index("d"))

### Multi-head attention

One head asks one kind of question. Real models run several in parallel, each with its own
projections, then concatenate the results and pass them through one more matrix `W_O`.

In [ ]:
rng = np.random.default_rng(5)
n_heads, d_model, d_head = 4, 4, 4

X = np.stack([Q[i] + K[i] for i in range(len(TOKENS))])   # a stand-in residual stream

heads = []
for h in range(n_heads):
    Wq, Wk, Wv = (rng.normal(0, 0.6, (d_model, d_head)) for _ in range(3))
    head_out, head_w, _ = attention(X @ Wq, X @ Wk, X @ Wv, causal=True)
    heads.append(head_out)
    top = TOKENS[int(head_w[it].argmax())]
    print(f"head {h}: with the mask on, 'it' looks most at '{top}'")

concat = np.concatenate(heads, axis=-1)
W_O = rng.normal(0, 0.4, (n_heads * d_head, d_model))
print(f"\nconcatenated: {concat.shape}  ->  after W_O: {(concat @ W_O).shape}")
print("Different random projections ask different questions. Training is what makes them useful.")

---
## Part 2. The real GPT-2

Now the part the browser could not do. In PyTorch we can ask the model for its attention weights
directly, and get one matrix per layer per head.

The next cell installs `transformers` if it is missing and downloads GPT-2 (about 500 MB the first
time). CPU is fine.

In [ ]:
try:
    import transformers  # noqa: F401
except ImportError:
    %pip install -q transformers torch

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, attn_implementation="eager")
model.eval()

print(f"{model_name}: {model.config.n_layer} layers, {model.config.n_head} heads, "
      f"d_model = {model.config.n_embd}, head width = {model.config.n_embd // model.config.n_head}")

> **Note.** `attn_implementation="eager"` matters. The faster fused kernels never build the
> attention matrix in memory, so they cannot hand it back - the same reason the browser export
> could not. To *see* attention you have to ask for the slow, explicit path.

In [ ]:
sentence = "Sarah fed the cat because it was hungry"

inputs = tokenizer(sentence, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attn = outputs.attentions                      # one tensor per layer
words = [tokenizer.decode([i]).strip() for i in inputs["input_ids"][0]]

print("tokens:", words)
print(f"layers: {len(attn)}, and each is {tuple(attn[0].shape)} = (batch, heads, query, key)")
print("\nEvery row of every one of those matrices sums to 1 - they are all softmaxes.")
print("check:", attn[0][0, 0].sum(dim=-1).round(decimals=4).tolist())

### Which head resolves the pronoun?

Our toy had one hand-built head that sent `it` to `cat`. GPT-2 has 144 heads and nobody told any of
them to do that. Let's search all of them for the ones that put the most weight on `cat` when the
query is `it`.

In [ ]:
it_pos = words.index("it")
cat_pos = words.index("cat")

found = []
for layer, a in enumerate(attn):
    for head in range(a.shape[1]):
        found.append((a[0, head, it_pos, cat_pos].item(), layer, head))

found.sort(reverse=True)
print(f"Searched {len(found)} heads. Top 5 for 'it' -> 'cat':\n")
for weight, layer, head in found[:5]:
    print(f"  layer {layer:2d}, head {head:2d}:  {weight * 100:5.1f}%")

best_w, best_layer, best_head = found[0]
print(f"\nNo one designed this. It fell out of predicting the next token.")

In [ ]:
import matplotlib.pyplot as plt


def plot_attention(matrix, words, title, ax):
    ax.imshow(matrix, cmap="BuGn", vmin=0, vmax=matrix.max())
    ax.set_xticks(range(len(words)))
    ax.set_yticks(range(len(words)))
    ax.set_xticklabels(words, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(words, fontsize=9)
    ax.set_xlabel("key (looked at)", fontsize=9)
    ax.set_ylabel("query (doing the looking)", fontsize=9)
    ax.set_title(title, fontsize=10)
    for i in range(len(words)):
        for j in range(len(words)):
            if matrix[i, j] > 0.12:
                ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center",
                        fontsize=7, color="white" if matrix[i, j] > 0.5 else "black")


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plot_attention(attn[0][0, 0].numpy(), words, "Layer 0, head 0 (early: often positional)", axes[0])
plot_attention(attn[best_layer][0, best_head].numpy(), words,
               f"Layer {best_layer}, head {best_head} (strongest 'it' -> 'cat')", axes[1])
plt.tight_layout()
plt.show()

print("Notice the lower-triangle shape in both: that is the causal mask, in a real model.")

### A wall of heads

One layer, all twelve heads. Most look busy and uninterpretable, a few have an obvious job. That
mix is the honest picture of what attention heads are actually like.

In [ ]:
layer_to_show = best_layer

fig, axes = plt.subplots(3, 4, figsize=(15, 11))
for head, ax in enumerate(axes.flat):
    ax.imshow(attn[layer_to_show][0, head].numpy(), cmap="BuGn", vmin=0, vmax=1)
    ax.set_title(f"head {head}", fontsize=9)
    ax.set_xticks(range(len(words)))
    ax.set_yticks(range(len(words)))
    ax.set_xticklabels(words, rotation=90, fontsize=6)
    ax.set_yticklabels(words, fontsize=6)

fig.suptitle(f"GPT-2, layer {layer_to_show}: all 12 heads on the same sentence", fontsize=12)
plt.tight_layout()
plt.show()

### Your turn

Change the sentence and re-run. A few that are worth a look:

- `"The trophy would not fit in the suitcase because it was too big"` - a classic ambiguous pronoun.
- `"The keys to the cabinet are on the table"` - does any head connect the verb back to `keys`
  rather than the nearer `cabinet`?
- `"Paris is to France as Rome is to"` - do any heads line up the two halves of the analogy?

Be sceptical of tidy stories, though. Attention weights show *where* information was read from,
not *what* the model concluded. It is evidence, not an explanation.

In [ ]:
your_sentence = "The trophy would not fit in the suitcase because it was too big"

inputs = tokenizer(your_sentence, return_tensors="pt")
with torch.no_grad():
    out2 = model(**inputs, output_attentions=True)
words2 = [tokenizer.decode([i]).strip() for i in inputs["input_ids"][0]]

fig, ax = plt.subplots(figsize=(8, 6.5))
plot_attention(out2.attentions[best_layer][0, best_head].numpy(), words2,
               f"Layer {best_layer}, head {best_head}", ax)
plt.tight_layout()
plt.show()

---
## What you just did

- Rebuilt the chapter's attention in NumPy and confirmed it reproduces `46.3%` exactly.
- Showed that a query really is `x @ W_Q`, one vector in and three out.
- Watched softmax saturate without the `sqrt(d)` division.
- Ran multi-head attention end to end.
- Pulled the true attention weights out of GPT-2, found the head that resolves a pronoun on its
  own, and plotted the causal mask as it appears in a real model.

Next: [Chapter 6](https://talktovishal.github.io/projects/llm-soup-to-nuts/chapter-6.html) takes
this exact machinery and asks what it costs to run - starting with those `present.N.key` tensors
you met in the browser.